In [ ]:
#Selection function

In [12]:
import matplotlib.pyplot as plt
import numpy as np
import math

In [13]:
occupied_center1 = {(-1, 0, 0),
 (0, -1, 0),
 (0, 0, -1),
 (0, 0, 0),
 (0, 1, -2),
 (0, 1, -1),
 (0, 1, 0),
 (1, 0, -1),
 (1, 0, 0),
 (2, 0, -1),
 (2, 0, 0),
 (2, 1, -1),
 (2, 2, -1),
 (2, 2, 0),
 (2, 3, -1),
 (3, 1, -1),
 (3, 2, 0)}

occupied_center2 = {(-1, 0, 0),
 (0, -1, 0),
 (0, 0, -1),
 (0, 0, 0),
 (0, 1, -2),
 (0, 1, -1),
 (0, 1, 0),
 (1, 0, -1),
 (1, 0, 0),
 (2, 0, -1),
 (2, 0, 0),
 (2, 1, -1),
 (2, 2, -1),
 (2, 2, 0),
 (2, 3, -1),
 (3, 1, -1),
 (3, 2, 1)}

In [ ]:
structures=[occupied_center1, occupied_center2] # give all lists containing occupied centers to the structures list

In [15]:
#function with both SAbyVol calculation and scoring

def SAbyVolScore(occupied_centers):

    volume = len(occupied_centers) #each cell has volume 1 
    
    surface_area = 0
    dirs = [
        (1, 0, 0), (-1, 0, 0),
        (0, 1, 0), (0, -1, 0),
        (0, 0, 1), (0, 0, -1),
    ]

    for x, y, z in occupied_centers:
        for dx, dy, dz in dirs:
            neighbor = (x + dx, y + dy, z + dz)

            if neighbor not in occupied_centers:
                surface_area += 1

     
    SAbyVol=surface_area / volume
    mass=len(occupied_centers) #mass is a function of number of cells
    exp_areabyVol= 1/(mass**(1/3)) #expected relationship  
    score_SAbyVol= abs(SAbyVol-exp_areabyVol) # absolute difference between the observed and expected values

    return surface_area / volume, score_SAbyVol

In [ ]:
#selecting the best structure based on SAbyVol score
SAVol_scores=[SAbyVolScore(structure)[0] for structure in structures]
print('SAVol_scores:', SAVol_scores)

max_SA = max(SAVol_scores)
print('max_SA:', max_SA)

scaled_SAs = [SA / max_SA for SA in SAVol_scores]
print('scaled_SAs:', scaled_SAs)

best_structure_index = scaled_SAs.index(max(scaled_SAs))
print('best_structure_index:', best_structure_index)

SAVol_scores: [3.764705882352941, 3.8823529411764706]
max_SA: 3.8823529411764706
scaled_SAs: [0.9696969696969697, 1.0]
best_structure_index: 1


In [17]:
#mechanical constraint function
def Structural_Constraint_1(occupied_centers):
    load = {}
    Load_Limit = 1
    for ox, oy, oz in occupied_centers:
        load[(ox, oy, oz)] = 1 #initializing every cube with a load equal to 1
    #sorting the cubes from largest to smallest (descending order) to easily find the base cubes
    sorted_centers = sorted(
        occupied_centers,
        key=lambda c:c[2],
        reverse=True
    )
    for centers in sorted_centers:
        x, y, z = centers
        supports =[]
        for sx in [-1,0,1]:
            for sy in [-1,0,1]:
                possibility = (
                    x+sx,
                    y+sy,
                    z-1
                )
                if possibility in occupied_centers:
                    supports.append(possibility)
            #print(supports)
        if not supports:
            continue
            #print(supports)
        portion = load[(x, y, z)]/len(supports) #equally distributes weight amongst all support cubes
        for support in supports:
            load[support] += portion #calculates how much load the base or support cubes experience
    #print(load)

    min_z = min(c[2] for c in occupied_centers)
    base_cubes = []
    acceptable = []
    for cube in occupied_centers:
        if cube[2] == min_z:
            base_cubes.append(cube)
            #print(ratio)
            if load[cube] <= Load_Limit:
                acceptable.append(cube)
    base_loads = [load[cube] for cube in base_cubes]
    a = len(acceptable)/len(base_cubes) #calculates proportion of base cubes under the stress limit
    b = np.average(base_loads)/max(base_loads)

    Structural_Constraint_1 = 0.5*(a+b)
    return Structural_Constraint_1

In [ ]:
#selecting the best structure based on mechanical stability score
mechStab_scores=[Structural_Constraint_1(structure) for structure in structures]
print('mechStab_scores:', mechStab_scores)

best_structure_index = mechStab_scores.index(max(mechStab_scores))
print('best_structure_index:', best_structure_index)

mechStab_scores: [np.float64(0.5), np.float64(0.5)]
best_structure_index: 0


In [ ]:
#selecting the best structure based on SAbyVol and mechanical stability scores
scoreProduct = [x * y for x, y in zip(SAVol_scores, mechStab_scores)]
print('scoreProduct:', scoreProduct)

max_product = max(scoreProduct)
print('max_product:', max_product)

best_structure_index = scoreProduct.index(max_product)
print('best_structure_index:', best_structure_index)



scoreProduct: [np.float64(1.8823529411764706), np.float64(1.9411764705882353)]
max_product: 1.9411764705882353
best_structure_index: 1
